# 16.2 图算法 / Graph Algorithms

**中文**：有了图，第一类要会的就是**遍历与路径**：怎么系统地走遍整张图？两点之间最短路怎么走？以及**中心性（centrality）**——在网络里"谁最重要"？这些是社交分析、路由、搜索引擎、反欺诈的基础。本节全部**从零实现**，再用 NetworkX 验证正确性。
**English**: With a graph in hand, the first essentials are **traversal and paths**: how to systematically visit the whole graph? What is the shortest path between two nodes? And **centrality** — "who matters most" in a network? These underpin social analysis, routing, search engines, and fraud detection. We implement everything **from scratch**, then verify against NetworkX.

---

**中文**：两种最基础的遍历：
**English**: Two most fundamental traversals:
- **广度优先 BFS（Breadth-First Search）**：像水波一样**一层一层**向外扩散——先访问所有 1 跳邻居，再 2 跳，再 3 跳……用**队列(FIFO)**实现。关键性质：在**无权图**里，BFS 第一次到达某节点的步数就是**最短路径长度**。
- **深度优先 DFS（Depth-First Search）**：一条路**走到黑**，走不动了再回退（回溯）——用**栈(LIFO)**或递归实现。常用于检测环、拓扑排序、连通分量。

**English**:
- **BFS (Breadth-First Search)**: spreads outward **layer by layer** like a ripple — visit all 1-hop neighbors, then 2-hop, then 3-hop… implemented with a **queue (FIFO)**. Key property: in an **unweighted** graph, the hop count when BFS first reaches a node is the **shortest path length**.
- **DFS (Depth-First Search)**: go **as deep as possible** down one path, then backtrack — implemented with a **stack (LIFO)** or recursion. Used for cycle detection, topological sort, connected components.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 算法面必考）**
> **中文**：**BFS 用队列、DFS 用栈/递归**，都是 $O(V+E)$。**无权图最短路 → BFS**；**带权(非负)最短路 → Dijkstra**(优先队列, $O((V+E)\log V)$)；**带负权 → Bellman-Ford**；**所有点对 → Floyd-Warshall $O(V^3)$**。**中心性**衡量节点重要性：度中心性(连得多)、接近中心性(离所有人都近)、**介数中心性(多少最短路经过你→"桥梁/掮客")**、特征向量中心性/PageRank(连接到重要节点)。
> **English**: **BFS uses a queue, DFS a stack/recursion**, both $O(V+E)$. **Unweighted shortest path → BFS**; **weighted (non-negative) → Dijkstra** (priority queue, $O((V+E)\log V)$); **negative weights → Bellman-Ford**; **all pairs → Floyd-Warshall $O(V^3)$**. **Centrality** measures node importance: degree (well-connected), closeness (close to everyone), **betweenness (how many shortest paths pass through you → "bridge/broker")**, eigenvector/PageRank (connected to important nodes).


In [ ]:

# ============================================================
# 数据 + 邻接表 / data + adjacency list
# 中文：遍历用邻接表最自然(每个节点存它的邻居集合)，稀疏图下高效。
# English: traversal is most natural on an adjacency list (neighbor set per node), efficient for sparse graphs.
# ============================================================
import networkx as nx, numpy as np, matplotlib.pyplot as plt
from collections import deque
np.random.seed(0)
G = nx.karate_club_graph()
adj = {u: set(G.neighbors(u)) for u in G.nodes()}         # 邻接表 / adjacency list
print("节点 / nodes:", G.number_of_nodes(), "| 边 / edges:", G.number_of_edges())
print("节点0的邻居 / neighbors of 0:", sorted(adj[0])[:8], "...")


**中文**：先手写 **BFS** 和 **DFS**，对比它们的访问顺序。BFS 用队列(从左边弹出、右边加入)，DFS 用栈(从右边弹出、右边加入)。
**English**: First hand-code **BFS** and **DFS** and compare their visit orders. BFS uses a queue (pop left, push right); DFS uses a stack (pop right, push right).


In [ ]:

# ============================================================
# 从零实现 BFS / DFS / BFS & DFS from scratch
# ============================================================
def bfs(adj, start):
    visited=[start]; seen={start}; q=deque([start])         # 队列初始化 / queue init
    while q:
        u=q.popleft()                                       # FIFO：从队首取 / pop front
        for v in sorted(adj[u]):                            # 遍历邻居(排序使结果确定) / neighbors
            if v not in seen:
                seen.add(v); visited.append(v); q.append(v) # 未访问则入队 / enqueue unseen
    return visited

def dfs(adj, start):
    visited=[]; seen=set(); stack=[start]                    # 栈初始化 / stack init
    while stack:
        u=stack.pop()                                        # LIFO：从栈顶取 / pop top
        if u in seen: continue
        seen.add(u); visited.append(u)
        for v in sorted(adj[u], reverse=True):               # 逆序压栈使小号先出 / push neighbors
            if v not in seen: stack.append(v)
    return visited

print("BFS from 0:", bfs(adj,0)[:12], "...")                 # 先近后远(按层) / by layers
print("DFS from 0:", dfs(adj,0)[:12], "...")                 # 一路到底 / dive deep
print("两者都访问全部 34 个节点 / both visit all:", len(bfs(adj,0))==34==len(dfs(adj,0)))


**中文**：BFS 的一个直接应用是**无权最短路**：在 BFS 时记录每个节点的"层数(距离)"和"父节点"，就能得到从起点到任意点的最短步数和具体路径。这正是社交网络里"你和某人相隔几度"的算法。
**English**: A direct use of BFS is the **unweighted shortest path**: while running BFS, record each node's "level (distance)" and "parent," giving the shortest hop count and the actual path from the source to any node. This is the algorithm behind "how many degrees of separation."


In [ ]:

# ============================================================
# BFS 最短路(无权) / BFS shortest path (unweighted)
# ============================================================
def bfs_shortest(adj, start):
    dist={start:0}; parent={start:None}; q=deque([start])
    while q:
        u=q.popleft()
        for v in sorted(adj[u]):
            if v not in dist:
                dist[v]=dist[u]+1                            # 层数+1 = 最短步数 / shortest hops
                parent[v]=u; q.append(v)
    return dist, parent

def reconstruct(parent, target):
    path=[];
    while target is not None: path.append(target); target=parent[target]
    return path[::-1]                                        # 从父指针回溯出路径 / trace back

dist, parent = bfs_shortest(adj, 0)
print("从 node0 到各点的最短步数(部分) / shortest hops from 0:",
      {k:dist[k] for k in [8,16,33,14]})
print("0 -> 16 的最短路径 / path:", reconstruct(parent,16))
# 用 NetworkX 验证 / verify against networkx
print("验证 verify (nx vs ours):",
      nx.shortest_path_length(G,0,16)==dist[16], nx.shortest_path(G,0,16)==reconstruct(parent,16))


**中文**：现实里边常常**带权**（距离、费用、时长）。带非负权的最短路要用 **Dijkstra 算法**：维护一个"已知最短距离"的优先队列，每次取出当前距离最小的节点，去**松弛(relax)** 它的邻居（如果经它中转更近就更新）。我们造一个带权小图来演示。
**English**: Real edges are often **weighted** (distance, cost, time). Shortest paths with non-negative weights use **Dijkstra's algorithm**: keep a priority queue of "best known distances," repeatedly pop the closest node, and **relax** its neighbors (update if going through it is shorter). We build a small weighted graph to demonstrate.


In [ ]:

# ============================================================
# Dijkstra 最短路(带权) / Dijkstra (weighted)
# ============================================================
import heapq
def dijkstra(wadj, start):
    dist={start:0.0}; pq=[(0.0,start)]; done=set()           # (距离,节点) 最小堆 / min-heap
    while pq:
        d,u=heapq.heappop(pq)                                # 取当前最近的节点 / closest node
        if u in done: continue                               # 已定型则跳过 / finalized
        done.add(u)
        for v,w in wadj[u]:                                  # 松弛邻居 / relax neighbors
            nd=d+w
            if nd < dist.get(v, float("inf")):               # 经 u 更近 → 更新 / shorter via u
                dist[v]=nd; heapq.heappush(pq,(nd,v))
    return dist

# 带权小图(城市间距离) / small weighted graph (city distances)
W={"A":[("B",1),("C",4)], "B":[("A",1),("C",2),("D",5)],
   "C":[("A",4),("B",2),("D",1)], "D":[("B",5),("C",1)]}
dd=dijkstra(W,"A")
print("从 A 出发的最短距离 / shortest distances from A:", dd)
print("A->D 最短=4 (A->B->C->D=1+2+1) 而非直连 / via B,C (=4) beats direct B->D")
# 用 networkx 验证 / verify
GW=nx.Graph();
for u,nb in W.items():
    for v,w in nb: GW.add_edge(u,v,weight=w)
print("验证 verify:", dd["D"]==nx.dijkstra_path_length(GW,"A","D"))


**中文**：现在进入更有"洞察"的部分——**中心性**：在网络里谁最重要？"重要"有多种定义，对应不同中心性：
**English**: Now the more insightful part — **centrality**: who matters most? "Important" has several definitions, each a different centrality:

**中文**：
- **度中心性 degree**：直接朋友多 = 重要。最简单、最局部。
- **接近中心性 closeness** = $\frac{n-1}{\sum_v d(u,v)}$：到所有人的平均距离短 = 重要（信息传播快）。
- **介数中心性 betweenness** = 有多少对节点的**最短路径要经过你** = 重要（你是"桥梁/掮客"，掌握咽喉）。
- **特征向量中心性 eigenvector**：连接到**重要的人**才算重要（递归定义，PageRank 的前身）。

**English**:
- **Degree**: many direct friends = important. Simplest, most local.
- **Closeness** = $\frac{n-1}{\sum_v d(u,v)}$: short average distance to everyone = important (fast information spread).
- **Betweenness** = how many node pairs' **shortest paths pass through you** = important (you are a "bridge/broker" controlling flow).
- **Eigenvector**: you are important if connected to **important** nodes (recursive definition, the precursor of PageRank).


In [ ]:

# ============================================================
# 从零算介数中心性(Brandes 简化版) + 对比四种中心性
# Betweenness from scratch (simplified Brandes) + compare four centralities
# ============================================================
def betweenness(adj):
    nodes=list(adj); bc={v:0.0 for v in nodes}
    for s in nodes:                                          # 以每个点为源做一次 BFS / BFS from each source
        S=[]; P={v:[] for v in nodes}; sigma={v:0 for v in nodes}; sigma[s]=1
        d={v:-1 for v in nodes}; d[s]=0; q=deque([s])
        while q:
            v=q.popleft(); S.append(v)
            for w in sorted(adj[v]):
                if d[w]<0: d[w]=d[v]+1; q.append(w)          # 首次到达 / first visit
                if d[w]==d[v]+1:                             # w 在 v 的下一层 / on a shortest path
                    sigma[w]+=sigma[v]; P[w].append(v)       # 累计最短路条数 / count shortest paths
        delta={v:0.0 for v in nodes}
        while S:                                             # 逆序回传依赖 / back-propagate dependencies
            w=S.pop()
            for v in P[w]:
                delta[v]+=(sigma[v]/sigma[w])*(1+delta[w])
            if w!=s: bc[w]+=delta[w]
    n=len(nodes); norm=(n-1)*(n-2)                           # 无向图归一化 / normalization
    return {v: bc[v]/norm for v in nodes}                    # (每对算了两次, 与 nx 默认一致用/2)

bc=betweenness(adj)                                          # Brandes 已对无向图给出与 nx 一致的结果 / matches nx
deg_c=nx.degree_centrality(G); clo_c=nx.closeness_centrality(G)
eig_c=nx.eigenvector_centrality(G,max_iter=1000); bc_nx=nx.betweenness_centrality(G)
print("我们的介数 vs networkx (node 0):", round(bc[0],4), "vs", round(bc_nx[0],4),
      "→ 一致" if abs(bc[0]-bc_nx[0])<1e-6 else "→ 不一致")
print("\n各中心性 Top-3 / top-3 by each centrality:")
for name,c in [("degree",deg_c),("closeness",clo_c),("betweenness",bc_nx),("eigenvector",eig_c)]:
    top=sorted(c.items(),key=lambda kv:kv[1],reverse=True)[:3]
    print(f"  {name:<12}", [(v,round(s,3)) for v,s in top])


**中文**：四种中心性给出的"重要节点"很值得对比——它们往往**不一致**，因为定义的"重要"不同。下面把介数中心性可视化(节点越大=越像桥梁)。
**English**: The four centralities' "important nodes" are worth comparing — they often **disagree** because they define "important" differently. Below we visualize betweenness (bigger node = more of a bridge).


In [ ]:

# ============================================================
# 可视化中心性 / visualize centralities
# ============================================================
pos=nx.spring_layout(G,seed=42)
fig,ax=plt.subplots(1,2,figsize=(15,6))
# ① 介数中心性 / betweenness
bsz=[bc_nx[i]*3000+50 for i in G.nodes()]
nd=nx.draw_networkx_nodes(G,pos,node_size=bsz,node_color=[bc_nx[i] for i in G.nodes()],cmap="plasma",ax=ax[0])
nx.draw_networkx_edges(G,pos,alpha=0.3,ax=ax[0]); nx.draw_networkx_labels(G,pos,font_size=8,ax=ax[0])
ax[0].set_title("介数中心性=桥梁(节点越大越关键) / betweenness = bridges"); ax[0].axis("off")
plt.colorbar(nd,ax=ax[0],fraction=0.04)
# ② 四中心性对各节点排名的对比(散点) / compare degree vs betweenness ranking
xs=[deg_c[i] for i in G.nodes()]; ys=[bc_nx[i] for i in G.nodes()]
ax[1].scatter(xs,ys,s=60,c=[eig_c[i] for i in G.nodes()],cmap="viridis")
for i in [0,33,32,2]: ax[1].annotate(str(i),(deg_c[i],bc_nx[i]),xytext=(5,5),textcoords="offset points")
ax[1].set_xlabel("degree centrality"); ax[1].set_ylabel("betweenness centrality")
ax[1].set_title("度高≠介数高(颜色=特征向量) / high degree ≠ high betweenness")
plt.tight_layout(); plt.savefig("/tmp/g02_viz.png",dpi=80); plt.show()
# 找一个"度中等却介数偏高"的掮客 / a broker: modest degree but high betweenness
deg=dict(G.degree())
cands=[i for i in G.nodes() if 2<=deg[i]<=5]                 # 度中等的候选 / mid-degree candidates
broker=max(cands, key=lambda i: bc_nx[i])
print(f"典型掮客节点 / broker node: {broker}  度仅 {deg[broker]} 但介数 {bc_nx[broker]:.3f}(连接不同区域)/ low degree, high betweenness")


**中文**：诚实解读：
**English**: Honest takeaways:

**中文**：
1. **不同中心性给出不同答案**：node 0 和 33（管理员、教练）在**所有**中心性上都高——它们既连得多、又是桥梁、又连着重要的人。但有些节点（如 node 0 与 33 之间的连接点）**度不算最高，介数却很高**——它们是连接两派的"咽喉"，删掉它们图最容易断成两半。
2. **介数中心性 = 找桥梁/瓶颈/掮客**：在反欺诈里是洗钱中转账户，在交通里是关键路口，在社交里是连接不同圈子的人——这是中心性里最有"洞察力"的一个。
3. **从零实现的 Brandes 算法和 NetworkX 完全一致**——理解它(BFS 累计最短路条数 + 逆序回传依赖)是图算法面试的高频考点。

**English**:
1. **Different centralities give different answers**: nodes 0 and 33 (admin, coach) score high on **all** — well-connected, bridging, and linked to important nodes. But some nodes have **modest degree yet high betweenness** — the "throats" connecting the two factions, whose removal most easily splits the graph.
2. **Betweenness = finding bridges/bottlenecks/brokers**: money-laundering relay accounts in fraud, key intersections in traffic, circle-connecting people in social nets — the most insightful centrality.
3. **Our from-scratch Brandes matches NetworkX exactly** — understanding it (BFS counting shortest paths + back-propagating dependencies) is a frequent graph-algorithm interview question.

> 💼 **实战视角 / Practical angle**
> **中文**：① **最短路**：地图导航(Dijkstra/A*)、网络路由、社交"几度好友"。② **中心性**：找 KOL(度/特征向量)、找关键基础设施与单点故障(介数)、反洗钱中转(介数)、搜索引擎权威度(特征向量→PageRank, 下节)。③ 大图上**精确介数 $O(VE)$ 太慢**，工业界用**采样近似**。面试金句：*"度看局部、接近看全局可达、介数看控制力、特征向量看声望——选哪个取决于你问的是哪种'重要'。"*
> **English**: ① **Shortest paths**: map navigation (Dijkstra/A*), routing, "degrees of friendship." ② **Centrality**: find KOLs (degree/eigenvector), critical infrastructure & single points of failure (betweenness), laundering relays (betweenness), search authority (eigenvector→PageRank, next section). ③ Exact betweenness is $O(VE)$ — too slow on big graphs, so industry uses **sampling approximations**. Interview line: *"Degree is local, closeness is global reach, betweenness is control, eigenvector is prestige — pick by which 'importance' you mean."*

---
### 小结 / Summary
- **中文**：BFS(队列,无权最短路)、DFS(栈,深探)；带权用 Dijkstra(优先队列)。
- **English**: BFS (queue, unweighted shortest path), DFS (stack, deep dive); weighted uses Dijkstra (priority queue).
- **中文**：四种中心性定义不同的"重要"——度/接近/介数/特征向量；介数最能找桥梁与瓶颈。
- **English**: Four centralities define "important" differently — degree/closeness/betweenness/eigenvector; betweenness best finds bridges and bottlenecks.
- **中文**：从零实现并用 NetworkX 验证，是吃透图算法的正确方式。
- **English**: Implementing from scratch and verifying against NetworkX is the right way to master graph algorithms.
